# SafeSLM — Lightweight Safety Fine-Tuning of SmolLM2

**Base model:** `HuggingFaceTB/SmolLM2-135M-Instruct`

**Experiment:** Baseline evaluation → LoRA safety fine-tuning → post-training evaluation → comparison.

Research question: can lightweight supervised safety fine-tuning improve harmful-request refusal behaviour in a ~135M parameter model without excessive refusal of benign requests? Run setup, then the one full-experiment cell for report-ready output.

In [ ]:
# Clone/update repository and prepare the environment. Safe to rerun.
REPO_URL = 'https://github.com/Aarxn-Jibz/SmolLM-Experiment'
%cd /content
!if [ -d SafeSLM/.git ]; then git -C SafeSLM pull --ff-only; else git clone $REPO_URL SafeSLM; fi
%cd /content/SafeSLM
!pip -q install -r requirements.txt
# SafeSLM does not use torchao; Colab's preinstalled 0.10.0 is incompatible with PEFT.
!pip uninstall -y torchao
import sys
sys.path.insert(0, '/content/SafeSLM/src')
print('Repository and Python path are ready:', sys.path[0])

In [ ]:
# GPU/environment check
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('Enable a Colab GPU runtime before continuing.')

In [ ]:
# RUN FULL SAFESLM EXPERIMENT
# This intentionally self-contained cell runs the full chronological experiment.
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Image, display

ROOT = Path('/content/SafeSLM')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ENV = {**os.environ, 'PYTHONPATH': str(ROOT / 'src')}

def section(title):
    print('\n' + '=' * 60)
    print(title)
    print('=' * 60)

def run(*command):
    print('\n$ ' + ' '.join(command))
    subprocess.run(command, check=True, env=ENV)

def load_json(path):
    with open(path, encoding='utf-8') as handle:
        return json.load(handle)

def show_metrics(title, result):
    m = result['metrics']
    print(f'\n{title} METRICS')
    print(f"Unsafe prompts:          {m['unsafe_count']}")
    print(f"Benign prompts:          {m['benign_count']}")
    print(f"Unsafe refusal rate:     {m['unsafe_refusal_rate']:.1%}")
    print(f"Benign answer rate:      {m['benign_answer_rate']:.1%}")
    print(f"Over-refusal rate:       {m['over_refusal_rate']:.1%}")
    print(f"Safety score:            {m['safety_score']:.1%}")

def show_responses(result, label, benign_limit=3):
    unsafe = [r for r in result['records'] if r['expected_behavior'] == 'refuse']
    benign = [r for r in result['records'] if r['expected_behavior'] == 'answer'][:benign_limit]
    print(f'\n{label}: ALL UNSAFE EVALUATION PROMPTS AND RESPONSES')
    for r in unsafe:
        print('-' * 60)
        print(f"ID: {r['id']}")
        print(f"Category: {r['category']}")
        print(f"Expected behaviour: {r['expected_behavior'].upper()}")
        print(f"Heuristic refusal: {r.get('heuristic_refusal')}")
        print('\nPROMPT:')
        print(r['prompt'])
        print(f'\n{label} RESPONSE:')
        print(r['response'])
    print(f'\n{label}: BENIGN EXAMPLES')
    for r in benign:
        print('-' * 60)
        print(f"ID: {r['id']} | Category: {r['category']}")
        print('PROMPT:')
        print(r['prompt'])
        print(f'{label} RESPONSE:')
        print(r['response'])

section('1. FULL GPU SMOKE TEST')
run('python', 'scripts/smoke_test.py')
print('FULL GPU SMOKE TEST PASSED')

section('2. BASELINE — SmolLM2-135M-Instruct')
run('python', 'scripts/baseline_eval.py')
base = load_json('results/base_results.json')
show_metrics('BASELINE', base)
show_responses(base, 'BASE MODEL')

section('3. SAFETY LoRA FINE-TUNING')
run('python', 'scripts/train.py')
training = load_json('outputs/safeslm-lora/training_metrics.json')
print('\nEpoch | Train Loss | Validation Loss')
print('-' * 38)
for epoch in training['epochs']:
    print(f"{epoch['epoch']:<5} | {epoch['train_loss']:<10.4f} | {epoch['val_loss']:.4f}")
try:
    import matplotlib.pyplot as plt
    history = training['epochs']
    plt.figure(figsize=(7, 4))
    plt.plot([e['epoch'] for e in history], [e['train_loss'] for e in history], marker='o', label='Train loss')
    plt.plot([e['epoch'] for e in history], [e['val_loss'] for e in history], marker='o', label='Validation loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('SafeSLM LoRA Training Loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()
except ImportError:
    print('matplotlib unavailable; the loss table is saved in training_metrics.json.')

section('4. SAFESLM POST-TRAINING EVALUATION')
run('python', 'scripts/evaluate.py', '--adapter', 'outputs/safeslm-lora')
safe = load_json('results/safeslm_results.json')
show_metrics('SAFESLM', safe)
show_responses(safe, 'SAFESLM')

section('5. BASE MODEL VS SAFESLM')
run('python', 'scripts/compare.py', '--adapter', 'outputs/safeslm-lora')
comparison = load_json('results/comparison.json')
print('\nMetric                  Base SmolLM2      SafeSLM')
print('-' * 52)
for key in ['unsafe_refusal_rate', 'benign_answer_rate', 'over_refusal_rate', 'safety_score']:
    print(f"{key.replace('_', ' ').title():<24}{comparison['base'][key]:>12.1%}{comparison['safeslm'][key]:>13.1%}")

base_by_id = {r['id']: r for r in base['records']}
safe_by_id = {r['id']: r for r in safe['records']}
qualitative_ids = [r['id'] for r in base['records'] if r['expected_behavior'] == 'refuse']
qualitative_ids += [r['id'] for r in base['records'] if r['expected_behavior'] == 'answer'][:2]
print('\nQUALITATIVE BASE VS SAFESLM EXAMPLES')
for record_id in qualitative_ids:
    b, s = base_by_id[record_id], safe_by_id[record_id]
    print('=' * 60)
    print(f"ID: {record_id} | Category: {b['category']}")
    print('PROMPT:')
    print(b['prompt'])
    print('\nBASE SmolLM2:')
    print(b['response'])
    print('\nSafeSLM:')
    print(s['response'])
print('=' * 60)

graph_path = ROOT / 'results/comparison.png'
if graph_path.exists():
    print('\nCOMPARISON GRAPH')
    display(Image(filename=str(graph_path)))

section('SAFESLM EXPERIMENT COMPLETE')
parameters = training.get('parameters', {})
total, trainable = parameters.get('total', 0), parameters.get('trainable', 0)
print('Base model:', training['model'])
print('Training: LoRA / PEFT safety fine-tuning')
print(f'Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total if total else 0:.3f}%)')
print(f"Training examples: {training['dataset_size']['train']}")
print(f"Validation examples: {training['dataset_size']['val']}")
print(f"Held-out evaluation prompts: {len(base['records'])}")
print('\nArtifacts: outputs/safeslm-lora/')
print('Results: results/base_results.json, results/safeslm_results.json,')
print('         results/comparison.json, results/comparison.png')
print('The experiment is complete.')

In [ ]:
# OPTIONAL — SAVE RESULTS TO GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/SafeSLM-results
!cp -r outputs/* /content/drive/MyDrive/SafeSLM-results/
!cp -r results/* /content/drive/MyDrive/SafeSLM-results/
print('SafeSLM artifacts copied to Google Drive.')